# 06-01 AutoGen 基础

**AutoGen**（微软）是另一个主流多 Agent 框架，核心思想：Agent 之间通过**对话**协作。

**本节目标**：ConversableAgent 基础、Agent 配置、双 Agent 对话

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from autogen import ConversableAgent
    HAS_AG = True
    print("AutoGen 导入成功")
except ImportError:
    HAS_AG = False
    print("AutoGen 未安装: pip install autogen-agentchat")

## 1. ConversableAgent

AutoGen 的核心抽象 —— 每个 Agent 都是一个 `ConversableAgent`，可以发送/接收消息。

```python
agent = ConversableAgent(
    name="ad_analyst",
    system_message="你是广告数据分析师",
    llm_config={"model": "gpt-4o-mini"},  # 或 None 表示人类 Agent
)
```

In [ ]:
# LLM 配置
llm_config = {
    "model": "gpt-4o-mini",
    "api_key": os.environ.get("OPENAI_API_KEY", "placeholder"),
    "temperature": 0.7,
}

if HAS_AG:
    # 创建广告分析师 Agent
    analyst = ConversableAgent(
        name="ad_analyst",
        system_message="""你是B站广告数据分析师。
职责：分析广告效果数据，提供数据洞察和优化建议。
回答简洁（3句话以内），用数据说话。""",
        llm_config={"config_list": [llm_config]},
    )
    
    # 创建广告主 Agent（模拟人类）
    advertiser = ConversableAgent(
        name="advertiser",
        system_message="你是游戏公司的广告主，关心CTR和ROI，会追问具体方案。",
        llm_config={"config_list": [llm_config]},
        max_consecutive_auto_reply=2,  # 最多自动回复2次
    )
    
    print("Agent 创建成功")
    print(f"  analyst: {analyst.name}")
    print(f"  advertiser: {advertiser.name}")
else:
    print("""
ConversableAgent 核心参数:
  name:            Agent 名称
  system_message:  系统提示（定义角色和行为）
  llm_config:      LLM 配置（None = 人类 Agent）
  max_consecutive_auto_reply: 最大连续自动回复次数（防止死循环）
  human_input_mode: "ALWAYS" / "TERMINATE" / "NEVER"
    """)

## 2. 双 Agent 对话

In [ ]:
if HAS_AG and os.environ.get("OPENAI_API_KEY"):
    # 发起对话
    result = advertiser.initiate_chat(
        analyst,
        message="我的游戏广告CTR只有1.2%，远低于行业均值，怎么优化？",
        max_turns=3,  # 最多3轮对话
    )
    
    print("\n=== 对话记录 ===")
    for msg in result.chat_history:
        print(f"[{msg.get('role', 'unknown')}] {msg['content'][:80]}...")
else:
    print("""
双 Agent 对话流程:
  result = agent_a.initiate_chat(
      agent_b,
      message="初始问题",
      max_turns=3,           # 最多对话轮数
  )

模拟对话:
  [advertiser] 我的游戏广告CTR只有1.2%，怎么优化？
  [analyst]    CTR低于均值2.1%，建议：1.优化创意素材 2.调整目标人群 3.测试投放时段
  [advertiser] 创意素材具体怎么优化？
  [analyst]    游戏广告建议：视频15s内，前3秒强吸引，加入游戏实机画面和限时优惠文案
    """)

## AutoGen vs LangGraph

| 特性 | AutoGen | LangGraph |
|------|---------|----------|
| 核心抽象 | Agent 对话 | 状态图 |
| 通信方式 | 消息传递 | 共享状态 |
| 控制流 | 对话轮次 | 图的边和条件 |
| 灵活性 | 对话场景好 | 复杂工作流好 |
| 持久化 | 需自行实现 | 内置 Checkpointer |
| 适用场景 | 多Agent讨论/辩论 | 生产级Agent编排 |

**下一节**: `02_autogen_tool_use.ipynb`